In [2]:
import pandas as pd
import numpy as np
from scipy.constants import pi

branch_data = pd.read_csv("branch.csv")
h_5 = 5 * 50

def create_z_5(branch_data):
    return [
        complex(row['r'], row['l'] * 2 * pi * h_5)
        for _, row in branch_data.iterrows()
    ]

def create_y_5(branch_data):
    return [
        complex(0, row['c'] * 2 * pi * h_5)
        for _, row in branch_data.iterrows()
    ]

def create_gammaL_5(branch_data, z_5, y_5):
    z_5 = np.array(z_5)
    y_5 = np.array(y_5)
    gammaL_5 = []
    for i, row in branch_data.iterrows():
        length = row['Length']
        gammaL = length * np.sqrt(z_5[i] * y_5[i])
        gammaL_5.append(gammaL)
    return gammaL_5

z_5 = create_z_5(branch_data)
y_5 = create_y_5(branch_data)
gammaL_5 = create_gammaL_5(branch_data, z_5, y_5)

def Z0_5(z_5, y_5):
    return np.sqrt(np.array(z_5) / np.array(y_5))

Z0_5 = Z0_5(z_5, y_5)
def ABCD_5(gammaL_5, Z0_5):
    A = np.cosh(gammaL_5)
    B = Z0_5 * np.sinh(gammaL_5)
    C = (1 / Z0_5) * np.sinh(gammaL_5)
    D = np.cosh(gammaL_5)
    return A, B, C, D
A_5, B_5, C_5, D_5 = ABCD_5(gammaL_5, Z0_5)
ABCD_5_df = pd.DataFrame({
    'From Bus Number': branch_data['From Bus  Number'],
    'To Bus Number'  : branch_data['To Bus  Number'],
    'A_5'           : A_5,
    'B_5'           : B_5,
    'C_5'           : C_5,
    'D_5'           : D_5
})

ABCD_5_df.to_csv("ABCD_5.csv", index=False)


In [3]:
print(z_5)

[(0.027104+1.4737792332630633j), (0.027104+1.4737792332630633j), (0.040967143+1.638686144038972j), (0.040967143+1.638686144038972j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.076053405+1.9348220923443065j), (0.076053405+1.9348220923443065j), (0.037948173+1.5299870382247651j), (0.037948173+1.5299870382247651j), (0.038033475+1.5299210647790398j), (0.038033475+1.5299210647790398j), (0.075983717+1.9360001895894028j), (0.075983717+1.9360001895894028j), (0.0121+0.726000071096026j), (0.038070732+1.5302069497105162j), (0.037470968+1.5300640072447782j), (0.075988+1.9360001895894028j), (0.075988+1.9360001895894028j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.0518848+1.479104232810898j), (0.0518848+1.479104232810898j), (0.016456+1.0260802889779426j), (0.016456+1.0260802889779426j), (0.016491852+1.0271562844617972j), (0.038236+1.5294404011030405j), (0.038236+1.5294404011030405j), (0.03

In [10]:
print(Z0_5)

[277.70676023-2.55341082j 277.70676023-2.55341082j
 275.42153283-3.44223051j 275.42153283-3.44223051j
 275.42181889-3.44223407j 275.42181889-3.44223407j
 360.09759024-7.07457223j 360.09759024-7.07457223j
 275.43529689-3.41527706j 275.43529689-3.41527706j
 275.44018084-3.42316j    275.44018084-3.42316j
 360.19145181-7.06563821j 360.19145181-7.06563821j
  45.20033511-0.37664327j 275.50242697-3.42664647j
 275.27667455-3.37023069j 360.19695535-7.06614415j
 360.19695535-7.06614415j 275.42181889-3.44223407j
 275.42181889-3.44223407j 275.42181889-3.44223407j
 275.42181889-3.44223407j 275.3660902 -4.82823379j
 275.3660902 -4.82823379j  54.24593888-0.43496293j
  54.24593888-0.43496293j  54.27474449-0.4356851j
 275.42181889-3.44223407j 275.42181889-3.44223407j
 275.4420077 -3.41950452j 275.4420077 -3.41950452j
 360.14266133-7.07014177j 360.14266133-7.07014177j
 275.41827745-3.44275682j 275.41827745-3.44275682j
 360.09499142-7.07507524j 360.09499142-7.07507524j
 275.42181889-3.44223407j 275.42181

In [11]:
import pandas as pd
import numpy as np

# Load and ensure numeric types
ABCD = pd.read_csv("ABCD_5.csv")

# Clean column names (optional, in case there are hidden spaces)
ABCD.columns = ABCD.columns.str.strip()

# Convert to complex numbers safely
for col in ['A_5', 'B_5', 'C_5', 'D_5']:
    ABCD[col] = ABCD[col].apply(lambda x: complex(x.replace('i', 'j')) if isinstance(x, str) else complex(x))

def combine_parallel(group):
    if len(group) == 1:
        row = group.iloc[0]
        return pd.Series({
            'A': row['A_5'],
            'B': row['B_5'],
            'C': row['C_5'],
            'D': row['D_5']
        })

    # Start with first line
    A_eq, B_eq, C_eq, D_eq = group.iloc[0][['A_5', 'B_5', 'C_5', 'D_5']]
    for _, row in group.iloc[1:].iterrows():
        A1, B1, C1, D1 = A_eq, B_eq, C_eq, D_eq
        A2, B2, C2, D2 = row['A_5'], row['B_5'], row['C_5'], row['D_5']

        A_eq = (A1 * B2 + A2 * B1) / (B1 + B2)
        B_eq = (B1 * B2) / (B1 + B2)
        C_eq = C1 + C2 + ((A1 - A2) * (D1 - D2)) / (B1 + B2)
        D_eq = (D1 * B2 + D2 * B1) / (B1 + B2)

    return pd.Series({
        'A': A_eq, 'B': B_eq, 'C': C_eq, 'D': D_eq
    })

# Combine lines with same (From, To)
ABCD_combined = (
    ABCD.groupby(['From Bus Number', 'To Bus Number'])
    .apply(combine_parallel)
    .reset_index()  # REMOVED drop=True so Pandas auto-restores the From/To columns
)

# Save result
ABCD_combined.to_csv("ABCD_parallel_combined_5.csv", index=False)

print(ABCD_combined.shape)

(27, 6)


C:\Users\User\AppData\Local\Temp\ipykernel_2764\2599097321.py:42: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(combine_parallel)


In [12]:
import pandas as pd
import numpy as np

# Load data
ABCD_combined = pd.read_csv("ABCD_parallel_combined.csv")

def abcd2s(df):
    S_data = []

    Z0 = 360.2607461227684
    Y0 = 1 / Z0

    for _, row in df.iterrows():
        from_bus = row["From Bus Number"]
        to_bus   = row["To Bus Number"]

        A = complex(row["A"])
        B = complex(row["B"])
        C = complex(row["C"])
        D = complex(row["D"])

        denom = (A + B*Y0 + C*Z0 + D)

        S11 = (A + B*Y0 - C*Z0 - D) / denom
        S12 = 2 * (A*D - B*C) / denom
        S21 = 2 / denom
        S22 = (-A + B*Y0 - C*Z0 + D) / denom

        S = np.array([
            [S11, S12],
            [S21, S22]
        ], dtype=complex)

        S_data.append({
            "from_bus": from_bus,
            "to_bus": to_bus,
            "S": S
        })

    return S_data
S_params = abcd2s(ABCD_combined)

for i, item in enumerate(S_params):
    print(f"\nLine {i}: Bus {item['from_bus']} → Bus {item['to_bus']}")
    print(item["S"])



Line 0: Bus 2135 → Bus 2220
[[-0.02353949-0.13114762j  0.97448622-0.17818419j]
 [ 0.97448622-0.17818419j -0.02353949-0.13114762j]]

Line 1: Bus 2135 → Bus 2400
[[-0.54056824-0.33097366j  0.3956228 -0.65689528j]
 [ 0.3956228 -0.65689528j -0.54056824-0.33097366j]]

Line 2: Bus 2135 → Bus 2970
[[-0.11785927-0.27322296j  0.8735043 -0.38158973j]
 [ 0.8735043 -0.38158973j -0.11785927-0.27322296j]]

Line 3: Bus 2220 → Bus 2225
[[-0.00822655-0.07379317j  0.98858779-0.12338457j]
 [ 0.98858779-0.12338457j -0.00822655-0.07379317j]]

Line 4: Bus 2220 → Bus 2230
[[-0.04405298-0.17722816j  0.9523104 -0.2411257j ]
 [ 0.9523104 -0.2411257j  -0.04405298-0.17722816j]]

Line 5: Bus 2220 → Bus 2570
[[-0.20462834-0.33361586j  0.78019381-0.48381641j]
 [ 0.78019381-0.48381641j -0.20462834-0.33361586j]]

Line 6: Bus 2220 → Bus 2691
[[-0.25270185-0.29857033j  0.68766833-0.60061615j]
 [ 0.68766833-0.60061615j -0.25270185-0.29857033j]]

Line 7: Bus 2222 → Bus 2300
[[-0.01609175-0.12395725j  0.98383294-0.1279877